# 06 — What does each codebook fragment represent?

The encoder maps every atom to one of 4096 codebook entries. To make sense
of generated histograms it helps to ask: *which atom environments does
fragment N actually correspond to in real molecules?*

This is the non-GTM half of the lab's `fragment_analysis.ipynb` — pure
Python, no proprietary GTM binary needed.

For the narrative see [`docs/fragment_inspection.md`](../docs/fragment_inspection.md).


## 1. Encode a small SDF + record per-molecule fragment ids

In [ ]:
import warnings
import numpy as np
from collections import defaultdict
from chython.files import SDFRead
from huggingface_hub import hf_hub_download
from VQGAE import VQGAE, vqgae_encode_dataset

warnings.filterwarnings("ignore", category=FutureWarning)

SDF = "../data/tubulin/colchicine_scaffold.sdf"
REPO = "tagirshin/VQGAE"

vqgae_enc = VQGAE.load_from_checkpoint(
    hf_hub_download(REPO, "vqgae.ckpt"),
    task="encode", batch_size=10, map_location="cpu",
).eval()

# Encode the whole SDF in one shot. Returns (N, max_atoms) int array,
# -1 padded; the same per-molecule slice you'd build with MolDataset + DataLoader.
codebook_inds = vqgae_encode_dataset(SDF, vqgae_enc, batch_size=10)
print(f"encoded {codebook_inds.shape[0]} molecules, max_atoms={codebook_inds.shape[1]}")

# Drop the padding to get one dict per molecule: {atom_index_1based: fragment_id}.
mols_frags = []
for row in codebook_inds:
    mols_frags.append({
        atom_i + 1: int(frag_i)
        for atom_i, frag_i in enumerate(row.tolist()) if frag_i >= 0
    })
print(f"per-molecule fragment-id maps built: {len(mols_frags)} entries")

## 2. Invert into `fragment_occurence`

In [ ]:
fragment_occurence = defaultdict(lambda: defaultdict(list))
for mol_i, mf in enumerate(mols_frags):
    for atom_i, frag_i in mf.items():
        fragment_occurence[frag_i][mol_i].append(atom_i)
fragment_occurence = {f: dict(v) for f, v in fragment_occurence.items()}
print(f"distinct codebook entries used: {len(fragment_occurence)} / 4096")

## 3. Atom-environment extractor

Walk each atom's bond graph outward and tabulate every shell as a sorted
tuple of `(symbol, hybridization, in_ring)`. Two atoms with the same
`env[:3]` (sphere depth 3) share a chemical context.


In [ ]:
def atom_env_extraction(mol, central_atom: int, max_env: int = 7):
    seen, env = set(), []
    frontier = [central_atom]
    for _ in range(max_env):
        shell = []
        next_frontier = []
        for n in frontier:
            if n in seen:
                continue
            atom = mol.atom(n)
            shell.append((atom.atomic_symbol, atom.hybridization, atom.in_ring))
            next_frontier.extend(mol._bonds[n])
        env.append(tuple(sorted(shell)))
        seen.update(frontier)
        frontier = next_frontier
    return tuple(env)

## 4. Pick the most-used fragment and inspect its environments

In [ ]:
ranked = sorted(
    fragment_occurence.items(),
    key=lambda kv: -sum(len(v) for v in kv[1].values()),
)
top_frag, top_mols = ranked[0]
n_total = sum(len(v) for v in top_mols.values())
print(f"fragment {top_frag} used by {len(top_mols)} molecules, {n_total} atoms total")

ENV_DEPTH = 3
unique = defaultdict(list)
with SDFRead(SDF, indexable=True) as inp:
    for mol_id, atom_ids in top_mols.items():
        molecule = inp[mol_id]
        env = atom_env_extraction(molecule, atom_ids[0])[:ENV_DEPTH]
        unique[env].append(mol_id)

unique = dict(sorted(unique.items(), key=lambda kv: -len(kv[1])))
print(f"  {len(unique)} distinct atom-environment patterns at depth {ENV_DEPTH}")
print()
for n, (pattern, mol_ids) in enumerate(unique.items()):
    if n >= 5:
        break
    pct = len(mol_ids) / n_total * 100
    print(f"  {len(mol_ids):>4d}  ({pct:5.1f} %)  {pattern}")

## 5. Try a few other fragments

Inspect a different codebook entry by changing the index in the next
cell. Re-run cells 4 and 5 each time.


In [ ]:
top_frag2, top_mols2 = ranked[1]
n2 = sum(len(v) for v in top_mols2.values())
print(f"fragment {top_frag2}: used by {len(top_mols2)} molecules, {n2} atoms")

ENV_DEPTH = 3
unique2 = defaultdict(list)
with SDFRead(SDF, indexable=True) as inp:
    for mol_id, atom_ids in top_mols2.items():
        env = atom_env_extraction(inp[mol_id], atom_ids[0])[:ENV_DEPTH]
        unique2[env].append(mol_id)

for pattern, mol_ids in sorted(unique2.items(), key=lambda kv: -len(kv[1]))[:5]:
    print(f"  {len(mol_ids):>4d}  {pattern}")

## What this is good for

- **Sanity-checking the codebook**: a fragment used by < 10 molecules in
  your data is long-tail; treat its appearances in generated candidates
  with caution.
- **Designing fragment masks**: when you fix a scaffold (tutorial 04),
  this is how you discover which codebook ids actually show up in the
  scaffold's atoms.
- **Diagnosing decode failures**: if `decode_population` drops too much,
  look at which fragment ids dominate the failures and inspect their
  environments — they may correspond to a hypervalent or charge-ambiguous
  atom that the decoder isn't trained on.

For the larger discussion see `docs/fragment_inspection.md`.
